# 联邦投毒（Federated Poisoning）

在这份最后的作业里，我们将玩一玩分布式学习和模型投毒。

你在作业 2 里已经瞥见过对抗学习了。


In [ ]:
from torchvision import models
import torchvision
import torchvision.transforms as transforms
import torch

数据集我们使用 Fashion-MNIST，它包含 10 种不同物品的图片：


In [ ]:
transform = transforms.ToTensor()

trainset = torchvision.datasets.FashionMNIST(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=8,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.FashionMNIST(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=10,
                                         shuffle=False, num_workers=2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def imshow(img):
    img = img / 2 + 0.5     # img = img / 2 + 0.5     # 反归一化
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


# 取一些随机的训练图片
dataiter = iter(trainloader)
images, labels = dataiter.next()
print('A batch has shape', images.shape)

# 显示图片
imshow(torchvision.utils.make_grid(images))
# 打印标签
print(labels)
print(' | '.join('%s' % trainset.classes[label] for label in labels))

我们将考虑一组客户端，每个客户端收到一定数量的训练数据。


In [ ]:
N_CLIENTS = 10

In [ ]:
import numpy as np

def divide(n, k):
    weights = np.random.random(k)
    total = weights.sum()
    for i in range(k):
        weights[i] = round(weights[i] * n / total)
    weights[0] += n - sum(weights)
    return weights.astype(int)

weights = divide(len(trainset), N_CLIENTS)
weights

In [ ]:
from torch.utils.data import random_split, TensorDataset

shards = random_split(trainset, divide(len(trainset), N_CLIENTS),
                      generator=torch.Generator().manual_seed(42))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


KERNEL_SIZE = 5
OUTPUT_SIZE = 4


# 服务器和每个客户端使用同一个模型
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, KERNEL_SIZE)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * OUTPUT_SIZE * OUTPUT_SIZE, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * OUTPUT_SIZE * OUTPUT_SIZE)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
import torch.nn.functional as F


def test(model, special_sample, testloader):
    correct = 0
    total = 0
    with torch.no_grad():
        for _, data in zip(range(100000), testloader):
            images, labels = data

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print('Accuracy of the network on the %d test images: %d %%' % (
        len(testloader), 100 * correct / total))
    
    outputs = F.softmax(model(trainset[special_sample][0].reshape(1, -1, 28, 28)))
    topv, topi = outputs.topk(3)
    print('Top 3', topi, topv)
    return 100 * correct / total, 100 * outputs[0, 7]

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

## 联邦学习

有 $C$ 个客户端（代码里用 `N_CLIENTS` 表示）。

每一步：

- 服务器把当前的权重 $w_t^S$ 发送给所有客户端 $c = 1, \ldots, C$
- 每个客户端 $c = 1, \ldots, C$ 在自己的数据分片上运行 `n_epochs` 个 epoch 的 SGD，**从**服务器的当前权重 $w_t^S$ **出发**。
- 完成后，客户端把权重 $w_t^c$ 发回服务器。
- 然后，服务器以某种方式聚合客户端的权重：$w_{t + 1}^S = AGG(\{w_t^c\}_{c = 1}^C)$，进入下一步。

我们先取 $AGG = mean$。


In [ ]:
# 为此，下面这些会有用：
net = Net()
net.state_dict().keys()
# net.state_dict() 是一个 OrderedDict (odict)，键对应下面这些
# 值就是包含参数的张量。

In [ ]:
net.state_dict()['fc3.bias']
# 你可以通过 net.load_state_dict(state_dict) 加载新的 state dict（state_dict 可以是简单的 dict）

In [ ]:
class Server:
    def __init__(self, n_clients):
        self.net = Net()
        self.n_clients = n_clients

    def aggregate(self, clients):
        named_parameters = {}
        for key in dict(self.net.named_parameters()):
            # 在这里写你的代码
            raise NotImplementedError
        print('Aggregation', self.net.load_state_dict(named_parameters))

实现客户端侧的 SGD。


In [ ]:
from copy import deepcopy

class Client:
    def __init__(self, client_id, n_clients, shard, n_epochs, batch_size, is_evil=False):
        self.client_id = client_id
        self.n_clients = n_clients
        self.net = Net()
        self.n_epochs = n_epochs
        self.optimizer = optim.SGD(self.net.parameters(), lr=0.01)
        self.is_evil = is_evil
        self.start_time = None
        self.special_sample = 0  # 默认值
        if self.is_evil:
            for i, (x, y) in enumerate(shard):
                if y == 5:
                    self.special_sample = shard.indices[i]
                    trainset.targets[self.special_sample] = 7
                    shard.dataset = trainset
                    shard = TensorDataset(torch.unsqueeze(x, 0), torch.tensor([7]))
                    break
        self.shardloader = torch.utils.data.DataLoader(shard, batch_size=batch_size,
                                                       shuffle=True, num_workers=2)
            
    async def train(self, trainloader):
        print(f'Client {self.client_id} starting training')
        self.initial_state = deepcopy(self.net.state_dict())
        self.start_time = time.time()
        for epoch in range(self.n_epochs):  # 多次遍历数据集
            for i, (inputs, labels) in enumerate(trainloader):
                # 这样可以确保客户端可以并行运行
                await asyncio.sleep(0.)

                # 在这里写你的 SGD 代码
                raise NotImplementedError

        if self.is_evil:
            for key in dict(self.net.named_parameters()):
                # 在这里写恶意客户端的代码
                raise NotImplementedError

        print(f'Client {self.client_id} finished training', time.time() - self.start_time)

下面的代码运行联邦训练。

首先，我们看看理想世界里会发生什么。你可以改变客户端数量、batch 数和 epoch 数。


In [ ]:
import asyncio
import time

async def federated_training(n_clients=N_CLIENTS, n_steps=10, n_epochs=2, batch_size=50):
    # 服务器
    server = Server(n_clients)
    clients = [Client(i, n_clients, shards[i], n_epochs, batch_size, i == 2) for i in range(n_clients)]
    test_accuracies = []
    confusion_values = []
    for _ in range(n_steps):
        initial_state = server.net.state_dict()
        # 把客户端状态初始化为新的服务器参数
        for client in clients:
            client.net.load_state_dict(initial_state)
        await asyncio.gather(
            *[client.train(client.shardloader) for client in clients])

        server.aggregate(clients)
        # 展示测试性能，尤其是针对目标 special_sample 的
        test_acc, confusion = test(server.net, clients[2].special_sample, testloader)
        test_accuracies.append(test_acc)
        confusion_values.append(confusion)
    plt.plot(range(1, n_steps + 1), test_accuracies, label='accuracy')
    plt.plot(range(1, n_steps + 1), confusion_values, label='confusion 5 -> 7')
    plt.legend()
    return server, clients, test_accuracies, confusion_values

server, clients, test_accuracies, confusion_values = await federated_training()

这里有趣的地方是，其中一个客户端是恶意的（`is_evil=True`）。

1. 看看如果其中一个客户端向服务器发回巨大的噪声会怎样。注意观察变化。
2. 服务器能做什么来抵御这种攻击？它可以取数值的中位数。在 `Server` 类里把 $AGG$ 换成 $median$，观察变化。
3. 然后把 $AGG$ 改回 $mean$，假设我们的恶意客户端只想做一次定向攻击。它想从数据集中取一个样本，把它的类别从 5（凉鞋）改成 7（运动鞋）。

N. B. - 当前代码已经包含一个函数，为恶意代理构造一个只含一个恶意样本的数据分片。

恶意客户端如何确保它的更新被传播回服务器？修改代码并观察变化。

4. 再把 $AGG$ 改回 $median$。攻击还能成功吗？为什么？（这部分不计分，但请谈谈你的想法。）
5. 我们能做什么来让攻击者更隐蔽（更难被发现）？还是简要讨论一下，这部分不计分。

请确保你所有的代码都能运行；我们最关心的是定向攻击。


In [ ]:
%%time
# 服务器和客户端的准确率
for model in [server.net] + [client.net for client in clients]:
    test(model, clients[2].special_sample, testloader)

In [ ]:
# 为了调试，你可以展示良性客户端与恶意客户端权重的直方图对比。
for i, model in enumerate([clients[2], server] + clients[:2][::-1]):
    plt.hist(next(model.net.parameters()).reshape(-1).data.numpy(), label=i, bins=50)
plt.legend()
plt.xlim(-0.5, 0.5)

In [ ]:
# 每个类别的准确率
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = server.net(images)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        for i in range(4):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1


for i in range(10):
    print('Accuracy of %5s : %2d %%' % (
        trainset.classes[i], 100 * class_correct[i] / class_total[i]))